# Compare Baseline CNN vs Residual CNN

This notebook loads the same noisy image, runs both trained denoising models, saves their outputs, and displays the results side by side for direct comparison.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch
from PIL import Image
import torchvision.transforms as T


def find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "models").exists() and (candidate / "training").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current notebook directory.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from models.baseline_cnn import BaselineCNN
from models.residual_cnn import ResidualCNN

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
print(f"Repository root: {REPO_ROOT}")

In [ ]:
def load_image(path: Path, image_size: int = 128) -> torch.Tensor:
    image = Image.open(path).convert("RGB")
    transform = T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
    ])
    return transform(image).unsqueeze(0)


def tensor_to_image(tensor: torch.Tensor) -> Image.Image:
    tensor = tensor.detach().cpu().clamp(0, 1).squeeze(0)
    return T.ToPILImage()(tensor)


def save_tensor_image(tensor: torch.Tensor, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tensor_to_image(tensor).save(path)


def load_model(model_name: str, checkpoint_path: Path) -> torch.nn.Module:
    if model_name == "baseline":
        model = BaselineCNN()
    elif model_name == "residual":
        model = ResidualCNN()
    else:
        raise ValueError(f"Unsupported model: {model_name}")

    state = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(state.get("model_state", state))
    model.to(DEVICE)
    model.eval()
    return model


def run_model(model_name: str, checkpoint_path: Path, noisy_tensor: torch.Tensor) -> torch.Tensor:
    model = load_model(model_name, checkpoint_path)
    with torch.no_grad():
        return model(noisy_tensor.to(DEVICE)).cpu()


def mean_absolute_difference(first: torch.Tensor, second: torch.Tensor) -> float:
    return torch.mean(torch.abs(first - second)).item()

In [ ]:
IMAGE_SIZE = 128
NOISY_IMAGE_PATH = REPO_ROOT / "data" / "noisy_test.png"
BASELINE_CHECKPOINT = REPO_ROOT / "training" / "baseline_cifar10_noise25.pth"
RESIDUAL_CHECKPOINT = REPO_ROOT / "training" / "residual_cifar10_noise25.pth"
BASELINE_OUTPUT = REPO_ROOT / "results" / "baseline_denoised.png"
RESIDUAL_OUTPUT = REPO_ROOT / "results" / "residual_denoised.png"

required_paths = [NOISY_IMAGE_PATH, BASELINE_CHECKPOINT, RESIDUAL_CHECKPOINT]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing_paths))

noisy_tensor = load_image(NOISY_IMAGE_PATH, IMAGE_SIZE)
baseline_output = run_model("baseline", BASELINE_CHECKPOINT, noisy_tensor)
residual_output = run_model("residual", RESIDUAL_CHECKPOINT, noisy_tensor)

save_tensor_image(baseline_output, BASELINE_OUTPUT)
save_tensor_image(residual_output, RESIDUAL_OUTPUT)

print(f"Saved baseline output to: {BASELINE_OUTPUT}")
print(f"Saved residual output to: {RESIDUAL_OUTPUT}")
print(f"Mean absolute difference between outputs: {mean_absolute_difference(baseline_output, residual_output):.6f}")

In [ ]:
comparison_images = [
    ("Noisy Input", tensor_to_image(noisy_tensor)),
    ("Baseline CNN Output", tensor_to_image(baseline_output)),
    ("Residual CNN Output", tensor_to_image(residual_output)),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for axis, (title, image) in zip(axes, comparison_images):
    axis.imshow(image)
    axis.set_title(title)
    axis.axis("off")

plt.tight_layout()
plt.show()